# Phase 4: Modelling (Part 3 - Hyperparameter Tuning)
**Client:** Crédit Nationale Azur
**Objective:** Use GridSearchCV to optimise the parameters of our "All Features" Random Forest and Support Vector Machine models to maximise the F1-Score.

In this notebook, we will recreate our strict data preparation pipeline on the complete dataset (bypassing feature selection) to deploy a systematic grid search across both algorithms.

In [10]:
# Required Imports for Tuning
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

## 1. Load Data and Prepare Targets
We load the clean data, map the target variable to binary integers, and perform our 80/20 stratified split to isolate our training data for the grid search.

In [11]:
# Load and map target
df = joblib.load('../data/cleaned_data.pkl')
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

# Split data (We will only feed X_train into the GridSearchCV)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()

## 2. One-Hot Encoding
We systematically encode all categorical text variables into integers.

In [12]:
# Identify and encode categorical columns
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train = pd.concat([X_train, dummies_train], axis=1).drop([col], axis=1)

print("Categorical encoding complete.")

Categorical encoding complete.


## 3. Standardisation
Because we are tuning a distance-based SVM, we must apply `StandardScaler` to all continuous features within our training set.

In [13]:
# Define and scale all continuous features
continuous_features = [
    'age', 'yrs_experience', 'family_size', 'income', 
    'mortgage_amt', 'credit_card_spend'
]

for col in continuous_features:
    scaler = StandardScaler()
    X_train[col] = scaler.fit_transform(X_train[col].values.reshape(-1, 1)).flatten()

print("Continuous features standardised. Ready for tuning.")

Continuous features standardised. Ready for tuning.


## 4. Tuning the Random Forest (All Features)
We will test different numbers of trees (`n_estimators`), tree depths (`max_depth`), and split criteria (`criterion`) using 5-fold cross-validation.

In [14]:
# Define the parameter grid
rf_param_grid = [
    {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'criterion': ['gini', 'entropy']
    }
]

# Instantiate and fit GridSearchCV
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    rf_param_grid, 
    cv=5, 
    scoring='f1', 
    verbose=1
)

print("Tuning Random Forest...")
rf_grid.fit(X_train, y_train)

print(f"\nBest RF Parameters: {rf_grid.best_params_}")
print(f"Best RF F1-Score: {rf_grid.best_score_:.4f}")

Tuning Random Forest...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best RF Parameters: {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 50}
Best RF F1-Score: 0.7245


## 5. Tuning the Support Vector Machine
As required, we use `np.arange` to generate float ranges for the regularisation parameter (`C`), alongside testing different kernels and class weights.

In [15]:
# Define the parameter grid using np.arange
svm_param_grid = [
    {
        'C': np.arange(0.5, 2.5, 0.5), 
        'kernel': ['linear', 'rbf'],
        'class_weight': [None, 'balanced']
    }
]

# Instantiate and fit GridSearchCV
svm_grid = GridSearchCV(
    SVC(random_state=42), 
    svm_param_grid, 
    cv=5, 
    scoring='f1', 
    verbose=1
)

print("Tuning Support Vector Machine...")
svm_grid.fit(X_train, y_train)

print(f"\nBest SVM Parameters: {svm_grid.best_params_}")
print(f"Best SVM F1-Score: {svm_grid.best_score_:.4f}")

Tuning Support Vector Machine...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best SVM Parameters: {'C': np.float64(1.5), 'class_weight': None, 'kernel': 'linear'}
Best SVM F1-Score: 0.7125
